In [ ]:
%%capture
# 1. Environment & Setup
!pip install transformers torch accelerate matplotlib numpy

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

# Detect device
device = 0 if torch.cuda.is_available() else -1
device_str = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device_str.upper()}\n")

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"

# =========================================================
# 2. Chapter 2/1: Tokenizers (Behind the Pipeline)
# =========================================================
print("===")
print("CHAPTER 2/1: TOKENIZERS")
print("===")

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
sample_text = "Using transformers library is both powerful and straightforward!"

# a) Subword Tokenization
tokens = tokenizer.tokenize(sample_text)
print(f"Sample Input: '{sample_text}'")
print(f"Tokens: {tokens}")

# b) Convert to IDs
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print(f"Token IDs: {token_ids}")

# c) Complete Tokenization with Special Tokens
inputs = tokenizer(sample_text, return_tensors="pt")
print(f"Input IDs Tensor:\n{inputs['input_ids']}")
print(f"Decoded Text with Special Tokens:\n'{tokenizer.decode(inputs['input_ids'][0])}'\n")


# =========================================================
# 3. Chapter 2/2: Models, Logits and Predictions
# =========================================================
print("===")
print("CHAPTER 2/2: MODELS & LOGITS")
print("===")

model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
if device_str == "cuda":
    model = model.to(device_str)
    inputs_to_model = {k: v.to(device_str) for k, v in inputs.items()}
else:
    inputs_to_model = inputs

# Inference
with torch.no_grad():
    outputs = model(**inputs_to_model)

logits = outputs.logits
print(f"Raw Logits: {logits.cpu().numpy()}")

# Logits to Probabilities (Softmax)
probabilities = F.softmax(logits, dim=-1)
pred_idx = torch.argmax(probabilities, dim=-1).item()
predicted_label = model.config.id2label[pred_idx]
confidence_score = probabilities[0, pred_idx].item()

print(f"Softmax Probabilities: {probabilities.cpu().numpy()}")
print(f"Predicted Class: {predicted_label} (Confidence: %{confidence_score * 100:.2f})\n")


# =========================================================
# 4. Chapter 2/3: Batching, Padding & Attention Mask
# =========================================================
print("===")
print("CHAPTER 2/3: BATCHING & ATTENTION MASK")
print("===")

batch_sentences = [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!"
]

batch_inputs = tokenizer(batch_sentences, padding=True, truncation=True, return_tensors="pt")
print(f"Batch Sentences: {batch_sentences}")
print(f"Padded Input IDs:\n{batch_inputs['input_ids']}")
print(f"Attention Mask:\n{batch_inputs['attention_mask']}\n")


# =========================================================
# 5. Chapter 2/4: Saving and Loading Models
# =========================================================
print("===")
print("CHAPTER 2/4: SAVING AND RELOADING")
print("===")

save_dir = "./saved_model_nlp02"
tokenizer.save_pretrained(save_dir)
model.save_pretrained(save_dir)
print(f"Tokenizer and Model saved locally to '{save_dir}'.")

# Reloading to verify
reloaded_tokenizer = AutoTokenizer.from_pretrained(save_dir)
reloaded_model = AutoModelForSequenceClassification.from_pretrained(save_dir)
print("Successfully reloaded model and tokenizer from local directory.\n")